In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms,datasets
import numpy as np
import torchvision.models as models
model=models.resnet50(weights="IMAGENET1K_V1")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 164MB/s]


In [2]:
SEED=42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
train_transform=transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2,hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.486,0.456,0.406],std=[0.229,0.224,0.225])
])
test_transform=transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.486,0.456,0.406],std=[0.229,0.224,0.225])
])
train_data=datasets.CIFAR10(root='./data',download=True,train=True,transform=train_transform)
test_data=datasets.CIFAR10(root='./data',download=True,train=False,transform=test_transform)

train_loader=DataLoader(train_data,batch_size=64,shuffle=True,pin_memory=True,num_workers=2)
test_loader=DataLoader(test_data,batch_size=64,shuffle=False,pin_memory=False,num_workers=2)

100%|██████████| 170M/170M [00:08<00:00, 20.8MB/s]


In [4]:
for param in model.layer4.parameters():
  param.requires_grad=True

In [5]:
optimizer=optim.Adam([
    {"params":model.layer4.parameters(),"lr":1e-4},
    {"params":model.fc.parameters(),"lr":1e-3}
])

In [6]:
criterion=nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler=optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=1)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)

In [7]:
epochs = 2
for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1)
        optimizer.step()

        train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    scheduler.step()

    print(f"\nEpoch [{epoch+1}/{epochs}]")
    print(f"Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%")


Epoch [1/2]
Train Loss: 2.1073, Train Acc: 59.86%

Epoch [2/2]
Train Loss: 1.8892, Train Acc: 66.65%


In [10]:
model.eval()
val_loss = 0
correct = 0
total = 0
with torch.no_grad():
  for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)

    outputs = model(images)
    loss = criterion(outputs, labels)
    val_loss += loss.item()
    _, predicted = torch.max(outputs, 1)
    correct += (predicted == labels).sum().item()
    total += labels.size(0)

  test_acc = 100 * correct / total
print(f"Val   Loss: {val_loss/len(test_loader):.4f}, Val   Acc: {test_acc:.2f}%")

Val   Loss: 1.4235, Val   Acc: 86.28%
